In [0]:
%run ./spark_config

In [0]:
# Notebook: 00_test_adls_connection
# Purpose: Verify Key Vault RBAC + all 5 ADLS containers working
# Safe to re-run anytime — does not write permanent data

# ── Load central config ───────────────────────────────────────────


print("=" * 55)
print(" ENVIRONMENT SETUP VERIFICATION")
print("=" * 55)

# ── Test 1: Key Vault RBAC secret retrieval ───────────────────────
print("\n📋 TEST 1: Key Vault RBAC Secret Retrieval")
print("-" * 55)

try:
    test_val = dbutils.secrets.get("portfolio-scope", "storage-account-name")
    print(f"✅ Key Vault RBAC connected — secret retrieved successfully")
    print(f"✅ Secret scope backend: Azure Key Vault")
except Exception as e:
    print(f"❌ Key Vault RBAC connection FAILED")
    print(f"   Error: {str(e)}")

# ── Test 2: All 5 ADLS container connections ──────────────────────
print("\n📋 TEST 2: ADLS Gen2 Container Connections")
print("-" * 55)

containers = ["staging", "bronze", "silver", "gold", "control"]
all_passed = True

for container in containers:
    path = f"abfss://{container}@{storage_acct}.dfs.core.windows.net/"
    try:
        dbutils.fs.ls(path)
        print(f"✅ {container:<12} — connected")
    except Exception as e:
        print(f"❌ {container:<12} — FAILED: {str(e)}")
        all_passed = False

# ── Test 3: Write + Read + Delete on staging ─────────────────────
print("\n📋 TEST 3: Write + Read + Delete on Staging")
print("-" * 55)

try:
    test_path = paths["staging"] + "_setup_test/test.txt"
    dbutils.fs.put(test_path, "rbac_connection_test_successful", overwrite=True)
    content   = dbutils.fs.head(test_path)
    dbutils.fs.rm(paths["staging"] + "_setup_test/", recurse=True)
    print(f"✅ Write  — successful")
    print(f"✅ Read   — successful")
    print(f"✅ Delete — successful")
except Exception as e:
    print(f"❌ Write/Read/Delete FAILED: {str(e)}")
    all_passed = False

# ── Test 4: Verify all table paths configured ─────────────────────
print("\n📋 TEST 4: Table Path Configuration")
print("-" * 55)

for name, path in table_paths.items():
    print(f"✅ {name:<25} → {path}")

# ── Final result ──────────────────────────────────────────────────
print("\n" + "=" * 55)
if all_passed:
    print("✅ ALL TESTS PASSED")
    print("✅ Azure Key Vault RBAC  — working")
    print("✅ ADLS Gen2 containers  — all 5 connected")
    print("✅ Service Principal     — authenticated")
    print("✅ Environment is production ready")
    print("\n🚀 Next: Build the data simulator notebook")
else:
    print("❌ SOME TESTS FAILED — fix errors above before proceeding")
print("=" * 55)